In [1]:
# Setup

In [1]:
%pip install pyspark==3.4.2 apache-sedona==1.6.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.1/311.1 MB 1.4 MB/s eta 0:00:0000:0100:04
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.0/179.0 kB 2.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 2.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.6/34.6 MB 2.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 2.1 MB/s eta 0:00:0000:0100:01
  Created wheel for pyspark: filename=pyspark-3.4.2-py2.py3-none-any.whl size=311619832 sha256=8fda08fd722bb4dd4593fbc5359abacfb5c69695c1821f41f791e63ebe5cfbba
  Stored in directory: /home/jovyan/.cache/pip/wheels/ae/78/cb/924eaddf18fb5bd07b68d76b6f706674e120883faa620a8d12
Successfully built pyspark
  Attempting uninstall: pyspark
    Found existing installation: pyspark 3.5.0
    Can't uninstall 'pyspark'. No files were found to uninstall.
Note: you may need to restart the kernel to use updated packages.


In [2]:
#MAPA
%pip install folium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.4/113.4 kB 1.9 MB/s eta 0:00:00 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import sys
from pathlib import Path
import urllib.request

# Setup Hadoop for Windows BEFORE importing pyspark
hadoop_home = Path.home() / "hadoop"
hadoop_bin = hadoop_home / "bin"
hadoop_bin.mkdir(parents=True, exist_ok=True)

winutils_path = hadoop_bin / "winutils.exe"
hadoop_dll_path = hadoop_bin / "hadoop.dll"

# Download winutils.exe and hadoop.dll if not present
files_to_download = [
    ("winutils.exe", "https://github.com/cdarlint/winutils/raw/master/hadoop-3.2.2/bin/winutils.exe"),
    ("hadoop.dll", "https://github.com/cdarlint/winutils/raw/master/hadoop-3.2.2/bin/hadoop.dll")
]

for filename, url in files_to_download:
    file_path = hadoop_bin / filename
    if not file_path.exists():
        print(f"Downloading {filename}...")
        try:
            urllib.request.urlretrieve(url, str(file_path))
            print(f"Downloaded {filename} to {file_path}")
        except Exception as e:
            print(f"Error downloading {filename}: {e}")

# Set environment variables BEFORE importing pyspark
os.environ['HADOOP_HOME'] = str(hadoop_home)
os.environ['PYSPARK_PYTHON'] = sys.executable

print(f"HADOOP_HOME set to: {os.environ['HADOOP_HOME']}")

# Now import pyspark
from pyspark.sql import SparkSession
import subprocess

subprocess.run(['where', 'java'], shell=True)

spark = (
    SparkSession.builder
    .appName("SedonaLocal")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.sedona:sedona-spark-shaded-3.4_2.12:1.6.1"
    )
    .config(
        "spark.sql.extensions",
        "org.apache.sedona.sql.SedonaSqlExtensions"
    )
    .config(
        "spark.serializer",
        "org.apache.spark.serializer.KryoSerializer"
    )
    .config(
        "spark.kryo.registrator",
        "org.apache.sedona.core.serde.SedonaKryoRegistrator"
    )
    .getOrCreate()
)

print("✓ Spark session created successfully!")

Downloaded winutils.exe to /home/jovyan/hadoop/bin/winutils.exe
Downloaded hadoop.dll to /home/jovyan/hadoop/bin/hadoop.dll
HADOOP_HOME set to: /home/jovyan/hadoop
✓ Spark session created successfully!


In [4]:
# Load Chicago crime data 2020
df = spark.read.option("header", True).csv("data/chicago/crimes/chicago_crimes_2020.csv")

In [5]:
# root me permite ver la primera fila con las columnas
df.printSchema()


root
 |-- id: string (nullable = true)
 |-- case_number: string (nullable = true)
 |-- date: string (nullable = true)
 |-- block: string (nullable = true)
 |-- iucr: string (nullable = true)
 |-- primary_type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- location_description: string (nullable = true)
 |-- arrest: string (nullable = true)
 |-- domestic: string (nullable = true)
 |-- beat: string (nullable = true)
 |-- district: string (nullable = true)
 |-- ward: string (nullable = true)
 |-- community_area: string (nullable = true)
 |-- fbi_code: string (nullable = true)
 |-- x_coordinate: string (nullable = true)
 |-- y_coordinate: string (nullable = true)
 |-- year: string (nullable = true)
 |-- updated_on: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- location: string (nullable = true)



In [6]:
from pyspark.sql.functions import col, expr

df_clean = (
    df
    .filter(col("Latitude").isNotNull())
    .filter(col("Longitude").isNotNull())
)


In [7]:
df_geo = df_clean.withColumn(
    "geometry",
    expr("ST_Point(cast(Longitude as double), cast(Latitude as double))")
)


In [8]:
df_geo.select("Latitude", "Longitude", "geometry").show(5, False)


+------------+-------------+----------------------------------+
|Latitude    |Longitude    |geometry                          |
+------------+-------------+----------------------------------+
|41.9116839  |-87.745917518|POINT (-87.745917518 41.9116839)  |
|41.863406877|-87.710455868|POINT (-87.710455868 41.863406877)|
|41.772893757|-87.619639151|POINT (-87.619639151 41.772893757)|
|41.888165132|-87.622937212|POINT (-87.622937212 41.888165132)|
|41.947024331|-87.748122241|POINT (-87.748122241 41.947024331)|
+------------+-------------+----------------------------------+
only showing top 5 rows



In [9]:
df_geo.createOrReplaceTempView("crimes")


In [10]:
spark.sql("""
    SELECT count(*) AS total,
           min(Year) AS min_year,
           max(Year) AS max_year
    FROM crimes
""").show()


+-----+--------+--------+
|total|min_year|max_year|
+-----+--------+--------+
|  941|    2020|    2020|
+-----+--------+--------+



In [12]:
# Prueba con reglas de asociacion apriori (no existe) pero tenemos fp-growth cumple la misma funcion y mas moderno

In [11]:
from pyspark.sql.functions import array, concat, lit, col

transactions = df_geo.select(
    array(
        concat(lit("TYPE="), col("primary_type")),
        concat(lit("LOCATION="), col("location_description")),
        concat(lit("ARREST="), col("arrest")),
        concat(lit("DOMESTIC="), col("domestic")),
        concat(lit("DISTRICT="), col("district"))
    ).alias("items")
)

In [12]:
from pyspark.ml.fpm import FPGrowth

fp = FPGrowth(
    itemsCol="items",
    minSupport=0.02,
    minConfidence=0.6
)

model = fp.fit(transactions)


In [13]:
model.associationRules.show(20, False)


+-------------------------------------------------------------+-------------------------+------------------+------------------+--------------------+
|antecedent                                                   |consequent               |confidence        |lift              |support             |
+-------------------------------------------------------------+-------------------------+------------------+------------------+--------------------+
|[TYPE=ROBBERY]                                               |[DOMESTIC=false]         |0.92              |1.184295485636115 |0.024442082890541977|
|[TYPE=ROBBERY]                                               |[ARREST=false]           |0.96              |1.13203007518797  |0.025504782146652496|
|[TYPE=OTHER OFFENSE, DOMESTIC=true]                          |[ARREST=false]           |0.9230769230769231|1.0884904569115097|0.025504782146652496|
|[DISTRICT=007]                                               |[DOMESTIC=false]         |0.673913043478260

In [14]:
# Clustering K-means

In [15]:
from pyspark.ml.feature import VectorAssembler

points = df_geo.select(
    col("Longitude").cast("double"),
    col("Latitude").cast("double")
)

assembler = VectorAssembler(
    inputCols=["Longitude", "Latitude"],
    outputCol="features"
)

dataset = assembler.transform(points)


In [16]:
from pyspark.ml.clustering import KMeans

kmeans = KMeans(k=8, seed=42)
model = kmeans.fit(dataset)


In [17]:
for i, c in enumerate(model.clusterCenters()):
    print(f"Cluster {i}: lon={c[0]}, lat={c[1]}")


Cluster 0: lon=-87.63501644073169, lat=41.789968780926856
Cluster 1: lon=-87.64888184535067, lat=41.89198025540299
Cluster 2: lon=-87.64697976308915, lat=41.70374039218812
Cluster 3: lon=-87.7813966744045, lat=41.951104994000005
Cluster 4: lon=-87.72117898132855, lat=41.786228442242844
Cluster 5: lon=-87.68139553350983, lat=41.98153275642156
Cluster 6: lon=-87.72824842881764, lat=41.889843957264716
Cluster 7: lon=-87.57686222297299, lat=41.74153530360359


In [18]:
# plot
sample = df_geo.select(
    col("Latitude").cast("double"),
    col("Longitude").cast("double")
).sample(fraction=0.01, seed=42)  # 1%


In [19]:
pdf = sample.toPandas()

In [20]:
import folium

m = folium.Map(
    location=[pdf.Latitude.mean(), pdf.Longitude.mean()],
    zoom_start=11
)

for _, row in pdf.iterrows():
    folium.CircleMarker(
        location=[row.Latitude, row.Longitude],
        radius=2,
        opacity=0.3
    ).add_to(m)

m


# Análisis de Relaciones: Crímenes, Estaciones de Policía y Transporte en Nueva York 2025

En esta sección analizaremos las relaciones espaciales entre:
- **Crímenes** registrados en Nueva York durante 2025
- **Estaciones de policía**
- **Estaciones de transporte**

Evaluaremos si existe correlación entre la ubicación de estas infraestructuras y la incidencia delictiva.

## 1. Carga de datos de Nueva York

In [ ]:
# Cargar datos NORMALIZADOS de crímenes de Nueva York (2025)
df_crimes_ny = spark.read.parquet("data/new_york/crimes_normalized/parquet").filter(col("year") == 2025)

# Cargar estaciones de policía
df_police = spark.read.option("header", True).csv("data/new_york/police_stations/new_york_police_stations.csv")

# Cargar estaciones de transporte (generado por data_download.ipynb)
import os
transport_path = "data/new_york/transport_stations/new_york_transport_stations.csv"
if os.path.exists(transport_path):
    df_transport = spark.read.option("header", True).csv(transport_path)
    print(f"Estaciones de transporte: {df_transport.count()} registros")
else:
    df_transport = None
    print(f"⚠ {transport_path} no encontrado. Ejecutar data_download.ipynb primero para descargarlo.")

print(f"Crímenes NY 2025: {df_crimes_ny.count()} registros")
print(f"Estaciones de policía: {df_police.count()} registros")

In [ ]:
# Ver estructura de los datos normalizados
df_crimes_ny.printSchema()
df_crimes_ny.select("offense_description", "latitude", "longitude", "crime_start_date").show(5, False)

## 2. Limpieza y preparación de datos geoespaciales

In [ ]:
# Los datos normalizados ya tienen tipos correctos (latitude/longitude como double)
# Solo filtramos coordenadas dentro del area de Nueva York
df_crimes_clean = (
    df_crimes_ny
    .filter(col("latitude").isNotNull())
    .filter(col("longitude").isNotNull())
    .filter((col("latitude") > 40.4) & (col("latitude") < 41.0))
    .filter((col("longitude") > -74.3) & (col("longitude") < -73.7))
)

print(f"Crímenes con coordenadas válidas: {df_crimes_clean.count()}")

# Crear geometría para crímenes
df_crimes_geo = df_crimes_clean.withColumn(
    "geometry",
    expr("ST_Point(longitude, latitude)")
)

In [26]:
# Filtrar estaciones de policía en el área de Nueva York
df_police_ny = (
    df_police
    .filter((col("lat").cast("double") > 40.4) & (col("lat").cast("double") < 41.0))
    .filter((col("lon").cast("double") > -74.3) & (col("lon").cast("double") < -73.7))
)

# Crear geometría para estaciones de policía
df_police_geo = df_police_ny.withColumn(
    "geometry",
    expr("ST_Point(cast(lon as double), cast(lat as double))")
)

print(f"Estaciones de policía en NY: {df_police_geo.count()}")

# Filtrar estaciones de transporte en el área de Nueva York
df_transport_ny = (
    df_transport
    .filter((col("lat").cast("double") > 40.4) & (col("lat").cast("double") < 41.0))
    .filter((col("lon").cast("double") > -74.3) & (col("lon").cast("double") < -73.7))
)

# Crear geometría para estaciones de transporte
df_transport_geo = df_transport_ny.withColumn(
    "geometry",
    expr("ST_Point(cast(lon as double), cast(lat as double))")
)

print(f"Estaciones de transporte en NY: {df_transport_geo.count()}")

Estaciones de policía en NY: 214
Estaciones de transporte en NY: 2811


## 3. Análisis de distancias y relaciones espaciales

In [ ]:
# Crear vistas temporales
df_crimes_geo.createOrReplaceTempView("crimes_ny")
df_police_geo.createOrReplaceTempView("police_stations")
df_transport_geo.createOrReplaceTempView("transport_stations")

# Calcular distancia de cada crimen a la estación de policía más cercana
query_police = """
SELECT 
    c.complaint_id,
    c.offense_description,
    c.latitude as crime_lat,
    c.longitude as crime_lon,
    MIN(ST_Distance(c.geometry, p.geometry) * 111.32) as dist_to_nearest_police_km,
    c.geometry as crime_geom
FROM crimes_ny c
CROSS JOIN police_stations p
GROUP BY c.complaint_id, c.offense_description, c.latitude, c.longitude, c.geometry
"""

df_crime_police_dist = spark.sql(query_police)
df_crime_police_dist.createOrReplaceTempView("crimes_with_police_dist")

In [ ]:
# Calcular distancia de cada crimen a la estación de transporte más cercana
query_transport = """
SELECT 
    c.complaint_id,
    c.offense_description,
    c.crime_lat,
    c.crime_lon,
    c.dist_to_nearest_police_km,
    MIN(ST_Distance(c.crime_geom, t.geometry) * 111.32) as dist_to_nearest_transport_km,
    c.crime_geom
FROM crimes_with_police_dist c
CROSS JOIN transport_stations t
GROUP BY c.complaint_id, c.offense_description, c.crime_lat, c.crime_lon, c.dist_to_nearest_police_km, c.crime_geom
"""

df_crime_distances = spark.sql(query_transport)
df_crime_distances.createOrReplaceTempView("crimes_with_distances")

In [ ]:
# Ver estadísticas de distancias
df_crime_distances.select(
    "offense_description", 
    "dist_to_nearest_police_km", 
    "dist_to_nearest_transport_km"
).describe().show()

# Ver algunos ejemplos
df_crime_distances.select(
    "offense_description",
    "dist_to_nearest_police_km",
    "dist_to_nearest_transport_km"
).show(10, False)

## 4. Categorización de cercanía y análisis de reglas de asociación

In [ ]:
# Categorizar proximidad a estaciones
df_categorized = df_crime_distances.withColumn(
    "near_police",
    expr("CASE WHEN dist_to_nearest_police_km < 0.5 THEN 'CERCA_POLICIA' ELSE 'LEJOS_POLICIA' END")
).withColumn(
    "near_transport",
    expr("CASE WHEN dist_to_nearest_transport_km < 0.5 THEN 'CERCA_TRANSPORTE' ELSE 'LEJOS_TRANSPORTE' END")
)

# Crear transacciones para FP-Growth con columnas normalizadas
transactions_ny = df_categorized.select(
    array(
        concat(lit("TIPO="), col("offense_description")),
        col("near_police"),
        col("near_transport")
    ).alias("items")
)

transactions_ny.show(10, False)

In [38]:
# Aplicar FP-Growth para encontrar patrones de asociación
fp_ny = FPGrowth(
    itemsCol="items",
    minSupport=0.01,
    minConfidence=0.8
)

model_ny = fp_ny.fit(transactions_ny)

# Ver patrones frecuentes
print("=== PATRONES FRECUENTES ===")
model_ny.freqItemsets.orderBy(col("freq").desc()).show(20, False)

print("\n=== REGLAS DE ASOCIACIÓN ===")
model_ny.associationRules.orderBy(col("confidence").desc()).show(20, False)

=== PATRONES FRECUENTES ===
+--------------------------------------------------------------------+----+
|items                                                               |freq|
+--------------------------------------------------------------------+----+
|[LEJOS_POLICIA]                                                     |945 |
|[LEJOS_TRANSPORTE]                                                  |910 |
|[LEJOS_TRANSPORTE, LEJOS_POLICIA]                                   |872 |
|[TIPO=HARRASSMENT 2]                                                |211 |
|[TIPO=HARRASSMENT 2, LEJOS_POLICIA]                                 |206 |
|[TIPO=HARRASSMENT 2, LEJOS_TRANSPORTE]                              |197 |
|[TIPO=HARRASSMENT 2, LEJOS_TRANSPORTE, LEJOS_POLICIA]               |194 |
|[TIPO=PETIT LARCENY]                                                |180 |
|[TIPO=PETIT LARCENY, LEJOS_POLICIA]                                 |172 |
|[TIPO=PETIT LARCENY, LEJOS_TRANSPORTE]                     

## 5. Análisis estadístico por tipo de crimen

In [ ]:
# Analizar relación entre tipo de crimen y distancia a estaciones
spark.sql("""
SELECT 
    offense_description as tipo_crimen,
    COUNT(*) as total_crimenes,
    ROUND(AVG(dist_to_nearest_police_km), 2) as dist_media_policia_km,
    ROUND(AVG(dist_to_nearest_transport_km), 2) as dist_media_transporte_km,
    SUM(CASE WHEN dist_to_nearest_police_km < 0.5 THEN 1 ELSE 0 END) as cerca_policia,
    SUM(CASE WHEN dist_to_nearest_transport_km < 0.5 THEN 1 ELSE 0 END) as cerca_transporte
FROM crimes_with_distances
GROUP BY offense_description
ORDER BY total_crimenes DESC
""").show(20, False)

## 6. Clustering espacial de crímenes

In [40]:
# Preparar datos para K-means
points_ny = df_crimes_geo.select(
    col("longitude").cast("double"),
    col("latitude").cast("double")
)

assembler_ny = VectorAssembler(
    inputCols=["longitude", "latitude"],
    outputCol="features"
)

dataset_ny = assembler_ny.transform(points_ny)

# Aplicar K-means con 10 clusters
kmeans_ny = KMeans(k=10, seed=42)
model_kmeans_ny = kmeans_ny.fit(dataset_ny)

# Predecir clusters
predictions_ny = model_kmeans_ny.transform(dataset_ny)

print("=== CENTROS DE CLUSTERS (ZONAS DE ALTA CRIMINALIDAD) ===")
for i, center in enumerate(model_kmeans_ny.clusterCenters()):
    print(f"Cluster {i}: lon={center[0]:.4f}, lat={center[1]:.4f}")

=== CENTROS DE CLUSTERS (ZONAS DE ALTA CRIMINALIDAD) ===
Cluster 0: lon=-74.1212, lat=40.6088
Cluster 1: lon=-74.1606, lat=40.5506
Cluster 2: lon=-74.2367, lat=40.5172
Cluster 3: lon=-74.1111, lat=40.5726
Cluster 4: lon=-74.1589, lat=40.6271
Cluster 5: lon=-73.8473, lat=40.8873
Cluster 6: lon=-73.9789, lat=40.7341
Cluster 7: lon=-74.1680, lat=40.5852
Cluster 8: lon=-74.1260, lat=40.6328
Cluster 9: lon=-74.2007, lat=40.5405


## 7. Visualización en mapa interactivo

In [ ]:
# Tomar muestras para visualización
sample_crimes = df_crimes_geo.select(
    "latitude",
    "longitude",
    "offense_description"
).sample(fraction=0.5, seed=42)

# Tomar todas las estaciones (son menos)
sample_police = df_police_geo.select(
    col("lat").cast("double").alias("latitude"),
    col("lon").cast("double").alias("longitude"),
    col("name")
).toPandas()

sample_transport = df_transport_geo.select(
    col("lat").cast("double").alias("latitude"),
    col("lon").cast("double").alias("longitude"),
    col("name"),
    col("transport_type")
).toPandas()

pdf_crimes = sample_crimes.toPandas()

print(f"Muestra de crímenes para mapa: {len(pdf_crimes)}")
print(f"Estaciones de policía: {len(sample_police)}")
print(f"Estaciones de transporte: {len(sample_transport)}")

In [ ]:
# Crear mapa centrado en Nueva York
m_ny = folium.Map(
    location=[40.7128, -74.0060],
    zoom_start=11,
    tiles='OpenStreetMap'
)

# Agregar centros de clusters de criminalidad
for i, center in enumerate(model_kmeans_ny.clusterCenters()):
    folium.Circle(
        location=[center[1], center[0]],
        radius=1000,
        color='red',
        fill=True,
        fillColor='red',
        fillOpacity=0.2,
        popup=f'Cluster criminalidad {i}',
        tooltip=f'Zona alta criminalidad {i}'
    ).add_to(m_ny)

# Agregar muestra de crímenes
for _, row in pdf_crimes.iterrows():
    folium.CircleMarker(
        location=[row.latitude, row.longitude],
        radius=2,
        color='darkred',
        fill=True,
        fillColor='red',
        fillOpacity=0.4,
        popup=row.offense_description,
        tooltip='Crimen'
    ).add_to(m_ny)

# Agregar estaciones de policía
for _, row in sample_police.iterrows():
    folium.Marker(
        location=[row.latitude, row.longitude],
        icon=folium.Icon(color='blue', icon='shield', prefix='fa'),
        popup=f"Policía: {row['name']}",
        tooltip='Estación de Policía'
    ).add_to(m_ny)

# Agregar estaciones de transporte
for _, row in sample_transport.iterrows():
    folium.Marker(
        location=[row.latitude, row.longitude],
        icon=folium.Icon(color='green', icon='train', prefix='fa'),
        popup=f"Transporte: {row['name']}<br>Tipo: {row['transport_type']}",
        tooltip='Estación de Transporte'
    ).add_to(m_ny)

# Agregar leyenda
legend_html = '''
<div style="position: fixed; 
            bottom: 50px; right: 50px; width: 250px; height: 180px; 
            background-color: white; border:2px solid grey; z-index:9999; 
            font-size:14px; padding: 10px">
<p><strong>Leyenda</strong></p>
<p><i class="fa fa-circle" style="color:red"></i> Crímenes (muestra)</p>
<p><i class="fa fa-circle" style="color:red;opacity:0.2"></i> Zonas alta criminalidad</p>
<p><i class="fa fa-shield" style="color:blue"></i> Estaciones de Policía</p>
<p><i class="fa fa-train" style="color:green"></i> Estaciones de Transporte</p>
</div>
'''
m_ny.get_root().html.add_child(folium.Element(legend_html))

# Mostrar mapa
m_ny

## 8. Análisis de correlación y conclusiones

In [43]:
# Análisis de proximidad: contar crímenes por categoría de distancia
spark.sql("""
SELECT 
    CASE 
        WHEN dist_to_nearest_police_km < 0.25 THEN 'Muy cerca (<250m)'
        WHEN dist_to_nearest_police_km < 0.5 THEN 'Cerca (250-500m)'
        WHEN dist_to_nearest_police_km < 1.0 THEN 'Distancia media (500m-1km)'
        ELSE 'Lejos (>1km)'
    END as proximidad_policia,
    CASE 
        WHEN dist_to_nearest_transport_km < 0.25 THEN 'Muy cerca (<250m)'
        WHEN dist_to_nearest_transport_km < 0.5 THEN 'Cerca (250-500m)'
        WHEN dist_to_nearest_transport_km < 1.0 THEN 'Distancia media (500m-1km)'
        ELSE 'Lejos (>1km)'
    END as proximidad_transporte,
    COUNT(*) as num_crimenes,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as porcentaje
FROM crimes_with_distances
GROUP BY 1, 2
ORDER BY num_crimenes DESC
""").show(20, False)

+--------------------------+--------------------------+------------+----------+
|proximidad_policia        |proximidad_transporte     |num_crimenes|porcentaje|
+--------------------------+--------------------------+------------+----------+
|Lejos (>1km)              |Lejos (>1km)              |691         |69.45     |
|Lejos (>1km)              |Distancia media (500m-1km)|108         |10.85     |
|Lejos (>1km)              |Cerca (250-500m)          |39          |3.92      |
|Distancia media (500m-1km)|Lejos (>1km)              |39          |3.92      |
|Distancia media (500m-1km)|Distancia media (500m-1km)|34          |3.42      |
|Lejos (>1km)              |Muy cerca (<250m)         |23          |2.31      |
|Muy cerca (<250m)         |Lejos (>1km)              |20          |2.01      |
|Muy cerca (<250m)         |Distancia media (500m-1km)|11          |1.11      |
|Distancia media (500m-1km)|Cerca (250-500m)          |6           |0.60      |
|Cerca (250-500m)          |Lejos (>1km)

In [44]:
# Resumen estadístico final
print("=" * 70)
print("RESUMEN DEL ANÁLISIS DE RELACIONES")
print("=" * 70)

# Contar crímenes cerca vs lejos
cerca_policia = df_categorized.filter(col("near_police") == "CERCA_POLICIA").count()
cerca_transporte = df_categorized.filter(col("near_transport") == "CERCA_TRANSPORTE").count()
total = df_categorized.count()

print(f"\nTotal de crímenes analizados: {total}")
print(f"\nCrímenes cerca de estaciones de policía (<500m): {cerca_policia} ({cerca_policia*100/total:.2f}%)")
print(f"Crímenes cerca de estaciones de transporte (<500m): {cerca_transporte} ({cerca_transporte*100/total:.2f}%)")

# Ambas condiciones
cerca_ambas = df_categorized.filter(
    (col("near_police") == "CERCA_POLICIA") & 
    (col("near_transport") == "CERCA_TRANSPORTE")
).count()

print(f"\nCrímenes cerca de AMBAS infraestructuras: {cerca_ambas} ({cerca_ambas*100/total:.2f}%)")

print("\n" + "=" * 70)

RESUMEN DEL ANÁLISIS DE RELACIONES

Total de crímenes analizados: 995

Crímenes cerca de estaciones de policía (<500m): 50 (5.03%)
Crímenes cerca de estaciones de transporte (<500m): 85 (8.54%)

Crímenes cerca de AMBAS infraestructuras: 12 (1.21%)

